# Auto-Labeling Pipeline

**Purpose:** Generate keyword-based labels for confident products in the **unlabeled pool**
(`backend/data/unlabeled/unlabeled_manifest.csv`, 54,922 codes) using the validated `classify_category()`
matcher, estimate label noise on random samples, and fold a chosen batch into a new v2 dataset that the
training notebook can ablate against the clean v1 baseline.

This notebook is a **data-generation step**, deliberately separate from `train_classifier.ipynb`:
- It runs independently (when the confidence threshold or `classify_category()` changes).
- It keeps the labeling logic decoupled from training logic so the ablation delta stays clean.
- The manual-validation step (human-in-the-loop eyeballs) lives here, not in a training loop.

## Outputs
- `backend/data/manifests/` — `train_manifest_v1_clean.csv`, `train_manifest_v2_autolabeled.csv`
- `backend/data/raw_v1_clean/<class>/` — hand-labeled images only (v1 baseline)
- `backend/data/raw_v2/<class>/` — hand-labeled images + chosen auto-labeled batch (v2)
- `backend/data/manifests/` reference CSVs for the candidate (conservative / loose) batches

## Ablation contract
`raw_v1_clean` and `raw_v2` differ **only** by the auto-labeled batch. Both exclude the ~5,789
pseudo-labeled strays already copied into `data/raw/` by `pseudo_label.py` (they are NOT in the
10,215-row manifest, so they are neither clean hand labels nor keyword auto-labels). The train-dev error
delta between the two therefore isolates the volume added by auto-labeling.

## Prerequisites
- Unlabeled manifest + images at `backend/data/unlabeled/` (`collect_unlabeled.py`)
- Labeled manifest at `backend/data/raw/manifest.csv` (`build_dataset.py`)
- HuggingFace `openfoodfacts/product-database` (split `beauty`) accessible for text lookup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm
from dotenv import load_dotenv

BACKEND_DIR = Path.cwd().parent
sys.path.insert(0, str(BACKEND_DIR))
load_dotenv(BACKEND_DIR / ".env")

from app.services.extraction import classify_category, CATEGORY_KEYWORDS

RAW_DIR = BACKEND_DIR / "data" / "raw"
UNLABELED_DIR = BACKEND_DIR / "data" / "unlabeled"
MANIFESTS_DIR = BACKEND_DIR / "data" / "manifests"
V1_CLEAN_DIR = BACKEND_DIR / "data" / "raw_v1_clean"
V2_DIR = BACKEND_DIR / "data" / "raw_v2"
V3_LOOSE_DIR = BACKEND_DIR / "data" / "raw_v3_loose"

for d in (MANIFESTS_DIR, V1_CLEAN_DIR, V2_DIR, V3_LOOSE_DIR):
    d.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = RAW_DIR / "manifest.csv"
UNLABELED_MANIFEST_PATH = UNLABELED_DIR / "unlabeled_manifest.csv"

print(f"Backend dir : {BACKEND_DIR}")
print(f"Raw manifest: {MANIFEST_PATH}")
print(f"Unlabeled   : {UNLABELED_MANIFEST_PATH}")
print(f"Keywords per class: {{k: len(v) for k, v in CATEGORY_KEYWORDS.items()}}")

Backend dir : c:\Projects\cosmetic-expiry-scanner\backend
Raw manifest: c:\Projects\cosmetic-expiry-scanner\backend\data\raw\manifest.csv
Unlabeled   : c:\Projects\cosmetic-expiry-scanner\backend\data\unlabeled\unlabeled_manifest.csv
Keywords per class: {k: len(v) for k, v in CATEGORY_KEYWORDS.items()}


In [3]:
def extract_text(field):
    """Extract English text from STRUCT(lang, text)[] field, fallback to first available."""
    if not isinstance(field, list) or len(field) == 0:
        return None
    for entry in field:
        if isinstance(entry, dict):
            lang = entry.get("lang", "")
            text = entry.get("text", "")
            if lang.startswith("en") and text:
                return text
    for entry in field:
        if isinstance(entry, dict) and entry.get("text"):
            return entry["text"]
    return None

## 0. Load the unlabeled pool and fetch product text

Stream through the HF beauty split, pulling `product_name` / `generic_name` for every unlabeled code.
This mirrors the lookup already proven in `validate_keyword_labeling.ipynb`.

In [4]:
unlabeled = pd.read_csv(UNLABELED_MANIFEST_PATH, dtype={"code": str})
print(f"Unlabeled manifest: {len(unlabeled):,} rows")
print(f"Distinct codes: {unlabeled['code'].nunique():,}")

print(f"\nImage availability on disk:")
img_map = {}
for p in UNLABELED_DIR.glob("*.jpg"):
    img_map[p.stem] = str(p)
unlabeled["image_exists"] = unlabeled["code"].isin(img_map)
print(unlabeled["image_exists"].value_counts().to_string())

Unlabeled manifest: 54,986 rows
Distinct codes: 54,986

Image availability on disk:
image_exists
True     54972
False       14


In [5]:
from datasets import load_dataset
from tqdm import tqdm

ds = load_dataset("openfoodfacts/product-database", split="beauty", streaming=True)

codes_needed = set(unlabeled["code"])
text_data = {}
remaining = set(codes_needed)

for row in tqdm(ds, desc="Fetching product text from HF"):
    code = str(row["code"])
    if code in remaining:
        text_data[code] = {
            "product_name_text": extract_text(row.get("product_name")),
            "generic_name_text": extract_text(row.get("generic_name")),
        }
        remaining.discard(code)
        if not remaining:
            break

print(f"Found text for {len(text_data):,} / {len(codes_needed):,} codes")
if remaining:
    print(f"Missing text (will be dropped): {len(remaining):,}")

text_df = pd.DataFrame.from_dict(text_data, orient="index").reset_index().rename(columns={"index": "code"})
pool = unlabeled.merge(text_df, on="code", how="left")
print(f"\nPool: {len(pool):,} rows")
print(f"product_name non-null: {pool['product_name_text'].notna().sum():,}")
print(f"generic_name non-null: {pool['generic_name_text'].notna().sum():,}")

Fetching product text from HF: 74359it [00:41, 1790.01it/s]


Found text for 54,986 / 54,986 codes

Pool: 54,986 rows
product_name non-null: 32,187
generic_name non-null: 2,699


## 1. Classify with confidence emulation

`classify_category()` returns only `(category, method)` with no numeric score, so we reconstruct which
keyword fired and from which priority pass. Each match is tagged **strong** (unambiguous product-type
word: shampoo, conditioner, mascara, lipstick, serum, sunscreen, ...) or **contextual** (descriptor /
ingredient / concern hit: hydrating, lifting, volume, créme, ...). We match against name with generic as
fallback, mirroring the prior validation notebook.

In [6]:
def match_detail(raw_text: str | None):
    """Return (category, keyword, is_strong) mirroring classify_category's pass order.

    Priority passes: 0 = haircare creme compounds, 1 = unambiguous makeup,
    2 = standard CATEGORY_KEYWORDS sweep.
    is_strong: keyword is an unambiguous product-type term (vs. contextual descriptor/ingredient).
    """
    if not raw_text:
        return None, None, None
    lower = raw_text.lower()

    HAIRCARE_CREME = {
        "creme colorante", "crème colorante",
        "creme decolorante", "crème décolorante", "creme décolorante",
        "creme de coiffage", "crème de coiffage",
        "creme fixante", "crème fixante",
    }
    for term in HAIRCARE_CREME:
        if term in lower:
            return "haircare", term, True

    MAKEUP_UNAMBIGUOUS = {
        "mascara", "eyeliner", "eye liner", "eyeshadow", "eye shadow",
        "lipstick", "lip gloss", "lip liner", "concealer", "foundation",
        "blush", "bronzer", "nail polish", "vernis à ongles", "vernis a ongles",
        "fond de teint",
    }
    for term in MAKEUP_UNAMBIGUOUS:
        if term in lower:
            return "makeup", term, True

    # Standard sweep. classify_category() early-returns on the first substring hit in
    # dict-insertion order, so a contextual descriptor (e.g. "hydrating") can shadow a
    # stronger product-type term later in the list (e.g. "face wash"). For CONFIDENCE
    # scoring we deliberately scan ALL keywords and prefer a strong term when one exists,
    # falling back to the first (category-consistent) hit otherwise. This is a confidence
    # refinement on top of classify_category(), not a change to it.
    best = (None, None, False)
    first_hit = None
    for category, keywords in CATEGORY_KEYWORDS.items():
        for kw in keywords:
            if kw in lower:
                is_strong = kw in STRONG_TERMS.get(category, set())
                if first_hit is None:
                    first_hit = (category, kw, is_strong)
                if is_strong and best[0] is None:
                    best = (category, kw, True)
    if best[0] is not None:
        return best
    return first_hit if first_hit is not None else (None, None, None)


# Strong (unambiguous product-type) terms per class. Contextual hits (descriptors, ingredients,
# concerns, ambiguous creme/creme) are intentionally excluded so the conservative batch stays clean.
STRONG_TERMS = {
    "skincare": {
        "serum", "essence", "ampoule", "booster", "sunscreen", "sunblock",
        "cleanser", "face wash", "micellar", "cleansing oil", "cleansing balm",
        "toner", "toning", "facial mist", "eye cream", "lip balm", "face mask",
        "sheet mask", "clay mask", "body lotion", "body cream", "body oil",
        "body butter", "hand cream", "shower gel", "body wash", "hand soap",
        "liquid soap", "bar soap", "shaving", "shaving foam", "shaving gel",
        "deodorant", "antiperspirant", "gel douche", "masque visage", "savon",
        "lotion corporelle", "creme mains", "soin corps", "lait corps",
        "nettoyant", "mousse à raser", "mousse a raser",
    },
    "haircare": {
        "shampoo", "conditioner", "co-wash", "cleansing conditioner",
        "hair mask", "hair treatment", "hair serum", "hair oil", "hair spray",
        "hairspray", "hair mousse", "styling mousse", "hair gel", "pomade",
        "hair cream", "leave-in", "hair wax", "hair paste", "scalp serum",
        "dandruff", "shampooing", "après-shampooing", "apres-shampoing",
        "masque cheveux", "huile cheveux", "gel coiffant", "sérum cheveux",
        "laque", "soin cheveux", "soins cheveux", "crème colorante",
    },
    "makeup": {
        "mascara", "eyeliner", "eye liner", "eyeshadow", "eye shadow",
        "lipstick", "lip gloss", "lip liner", "concealer", "foundation",
        "blush", "bronzer", "nail polish", "vernis à ongles", "vernis a ongles",
        "fond de teint", "bb cream", "cc cream", "primer", "highlighter",
        "setting spray", "setting powder", "pressed powder", "loose powder",
        "eyebrow", "brow gel", "brow pencil", "lip plumper", "lip stain",
        "false lashes", "maquillage", "rouge à lèvres", "rouge a levres",
        "fard à paupières", "fard a paupieres", "anticernes", "correcteur",
        "gloss", "micellar water", "makeup remover", "cleansing wipe",
    },
}

pool["pred_combined"] = pool["product_name_text"].fillna(pool["generic_name_text"])
pool[["pred_name", "keyword_name", "strong_name"]] = pool[
    "product_name_text"
].apply(lambda t: pd.Series(match_detail(t) if isinstance(t, str) else (None, None, None)))
pool[["pred_generic", "keyword_generic", "strong_generic"]] = pool[
    "generic_name_text"
].apply(lambda t: pd.Series(match_detail(t) if isinstance(t, str) else (None, None, None)))

# Combined: prefer a strong name match, else generic; fall back to any name/generic prediction.
def combine(row):
    if pd.notna(row["pred_name"]):
        return pd.Series([row["pred_name"], row["keyword_name"], bool(row["strong_name"])],
                         index=["pred", "keyword", "strong"])
    if pd.notna(row["pred_generic"]):
        return pd.Series([row["pred_generic"], row["keyword_generic"], bool(row["strong_generic"])],
                         index=["pred", "keyword", "strong"])
    return pd.Series([None, None, False], index=["pred", "keyword", "strong"])

pool[["pred", "keyword", "strong"]] = pool.apply(combine, axis=1)

print("Combined predictions:")
print(pool["pred"].value_counts(dropna=False).to_string())
print(f"\nMatched (has prediction): {pool['pred'].notna().sum():,}")
print(f"Strong matches: {pool['pred'].notna() & pool['strong']} -> {int((pool['pred'].notna() & pool['strong']).sum()):,}")

Combined predictions:
pred
NaN         44388
skincare     7066
haircare     2317
makeup       1215

Matched (has prediction): 10,598
Strong matches: 0        False
1        False
2        False
3        False
4        False
         ...  
54981    False
54982    False
54983    False
54984    False
54985    False
Length: 54986, dtype: bool -> 6,184


## 2. Emit two candidate batches

- **Conservative**: only `strong` (unambiguous) single-category hits.
- **Loose**: all single-category hits including contextual/descriptor matches.

Both require an image to exist on disk so they can actually be folded into training later. Rows with no
image are reported but excluded from the batches.

In [7]:
matched = pool[pool["pred"].notna()].copy()
print(f"All matches: {len(matched):,}")
print(f"  with image on disk: {matched['image_exists'].sum():,}")

loose = matched[matched["image_exists"]].copy()
conservative = matched[matched["strong"] & matched["image_exists"]].copy()

def summarize(df, name):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"Volume: {len(df):,}")
    print("Per-class:")
    vc = df["pred"].value_counts()
    for c in ["skincare", "haircare", "makeup"]:
        print(f"  {c}: {int(vc.get(c, 0)):,}")

summarize(loose, "LOOSE batch (strong + contextual)")
summarize(conservative, "CONSERVATIVE batch (strong only)")

All matches: 10,598
  with image on disk: 10,592

  LOOSE batch (strong + contextual)
Volume: 10,592
Per-class:
  skincare: 7,061
  haircare: 2,317
  makeup: 1,214

  CONSERVATIVE batch (strong only)
Volume: 6,179
Per-class:
  skincare: 3,319
  haircare: 1,873
  makeup: 987


In [8]:
# Persist reference copies of both candidate batches for later re-decision.
sel_cols = ["code", "imgid", "pred", "keyword", "strong", "product_name_text", "generic_name_text"]
loose[sel_cols].to_csv(MANIFESTS_DIR / "candidate_loose.csv", index=False)
conservative[sel_cols].to_csv(MANIFESTS_DIR / "candidate_conservative.csv", index=False)
print(f"Wrote candidate CSVs to {MANIFESTS_DIR}")

Wrote candidate CSVs to c:\Projects\cosmetic-expiry-scanner\backend\data\manifests


## 3. Manual validation: 300/batch, class-stratified random sample

Draw a **true random, class-stratified** sample from each candidate batch (not just the easy hits) so the
noise estimate isn't biased toward easy cases. Review them in the browser app (next cell starts a local
server): each row shows the product image + name + predicted category, and you label with keys `1`/`2`/`3`
(`0`/space to skip, arrows to move). Click **Save to CSV** when done. The annotated CSVs land in
`data/manifests/manual_review_<batch>_annotated.csv`.

**This is the decision data**: pick the threshold based on noise rate AND volume AND per-class spread
(a "90% accurate but all in one class" batch isn't obviously better than "82% but 4,000 spread across
all three").

In [10]:
SAMPLE_PER_BATCH = 300
RNG_SEED = 42


def stratified_sample(df, n, seed):
    """Class-stratified random sample (proportional to class frequency in df)."""
    rng = np.random.default_rng(seed)
    counts = df["pred"].value_counts()
    parts = []
    for cls in counts.index:
        cls_df = df[df["pred"] == cls]
        k = int(round(n * counts[cls] / len(df)))
        k = max(1, min(k, len(cls_df)))
        parts.append(cls_df.sample(k, random_state=int(rng.integers(0, 2**32 - 1))))
    out = pd.concat(parts, ignore_index=True)
    if len(out) < n:
        leftover = df.drop(out.index)
        if len(leftover) > 0:
            extra = leftover.sample(min(n - len(out), len(leftover)), random_state=seed)
            out = pd.concat([out, extra], ignore_index=True)
    return out


sample_loose = stratified_sample(loose, SAMPLE_PER_BATCH, RNG_SEED)
sample_cons = stratified_sample(conservative, SAMPLE_PER_BATCH, RNG_SEED)

print("LOOSE sample size:", len(sample_loose))
print(sample_loose["pred"].value_counts().to_string())
print("\nCONSERVATIVE sample size:", len(sample_cons))
print(sample_cons["pred"].value_counts().to_string())

LOOSE sample size: 300
pred
skincare    200
haircare     66
makeup       34

CONSERVATIVE sample size: 300
pred
skincare    161
haircare     91
makeup       48


In [11]:
# Export the review samples. The CSV carries an empty manual_label column for spreadsheet
# editing; the JSON feeds the in-browser review app (Option A — next cell starts the server).
import json as _json

review_cols = ["code", "pred", "keyword", "product_name_text", "generic_name_text"]
for batch_name, sample in [("loose", sample_loose), ("conservative", sample_cons)]:
    df = sample[review_cols].copy()
    df["manual_label"] = ""
    df.to_csv(MANIFESTS_DIR / f"manual_review_{batch_name}.csv", index=False)
    rows_out = []
    for _, r in df.iterrows():
        img = UNLABELED_DIR / f"{r['code']}.jpg"
        rows_out.append({
            "code": r["code"],
            "pred": r["pred"],
            "keyword": r["keyword"],
            "name": None if pd.isna(r["product_name_text"]) else str(r["product_name_text"]),
            "generic": None if pd.isna(r["generic_name_text"]) else str(r["generic_name_text"]),
            "image": f"/data/unlabeled/{r['code']}.jpg" if img.exists() else None,
        })
    with open(MANIFESTS_DIR / f"manual_review_{batch_name}.json", "w", encoding="utf-8") as f:
        _json.dump({"batch": batch_name, "rows": rows_out}, f, ensure_ascii=False)
    print(f"  {batch_name}: {len(df):,} rows -> data/manifests/manual_review_{batch_name}.{{csv,json}}")

print("\nReview in the browser via the next cell (starts the local review server).")

  loose: 300 rows -> data/manifests/manual_review_loose.{csv,json}
  conservative: 300 rows -> data/manifests/manual_review_conservative.{csv,json}

Review in the browser via the next cell (starts the local review server).


In [12]:
# Start the local review server (backend/review/review_server.py) on a background thread.
# Then open one (or both) review URLs in your browser:
#   http://127.0.0.1:8787/review?batch=loose        -> LOOSE batch
#   http://127.0.0.1:8787/review?batch=conservative -> CONSERVATIVE batch
# Label each product with the 1/2/3 keys (image + name shown), click Save to CSV when done.
import sys as _sys
import threading as _threading

_REVIEW_DIR = BACKEND_DIR / "review"
_sys.path.insert(0, str(_REVIEW_DIR))
from review_server import start_server

REVIEW_PORT = 8787


def _ensure_review_server():
    if globals().get("_review_httpd") is not None:
        return True
    try:
        httpd = start_server(REVIEW_PORT)
    except OSError as e:
        # already running (e.g. from a terminal or a previous kernel) — fine
        if e.errno in (98, 10048):
            return True
        raise
    globals()["_review_httpd"] = httpd
    _threading.Thread(target=httpd.serve_forever, daemon=True).start()
    return True


_ensure_review_server()
print("Review server running. Open:")
print(f"  http://127.0.0.1:{REVIEW_PORT}/review?batch=loose")
print(f"  http://127.0.0.1:{REVIEW_PORT}/review?batch=conservative")

Review server running. Open:
  http://127.0.0.1:8787/review?batch=loose
  http://127.0.0.1:8787/review?batch=conservative


In [13]:
# Load your annotated labels back in. The base review CSV (manual_review_<batch>.csv)
# carries the prediction columns; the review app supplements manual labels in
# manual_review_<batch>_annotated.csv. Assemble a single frame with both, so
# noise_report can compare pred vs manual_label.
def _clean_label(v):
    if not isinstance(v, str):
        return None
    v = v.strip().lower()
    return v if v in ("skincare", "haircare", "makeup") else None


def load_review(batch):
    base = MANIFESTS_DIR / f"manual_review_{batch}.csv"
    if not base.exists():
        return None
    df = pd.read_csv(base, dtype={"code": str})
    if "manual_label" not in df.columns:
        df["manual_label"] = np.nan
    df["manual_label"] = df["manual_label"].astype(object).apply(_clean_label)
    annotated = MANIFESTS_DIR / f"manual_review_{batch}_annotated.csv"
    if annotated.exists():
        ann = pd.read_csv(annotated, dtype={"code": str})
        ann["manual_label"] = ann["manual_label"].astype(object).apply(_clean_label)
        df = df.drop(columns="manual_label").merge(
            ann[["code", "manual_label"]], on="code", how="left"
        )
    return df


review_loose = load_review("loose")
review_cons = load_review("conservative")

for name, df in [("LOOSE", review_loose), ("CONSERVATIVE", review_cons)]:
    if df is None:
        print(f"{name}: no base review CSV found.")
    else:
        n_lab = int(df["manual_label"].notna().sum())
        print(f"{name}: {len(df):,} rows, {n_lab:,} labeled")

LOOSE: 300 rows, 93 labeled
CONSERVATIVE: 300 rows, 168 labeled


In [14]:
def wilson_ci(correct, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = correct / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2))) / denom
    return (max(0.0, center - half), min(1.0, center + half))


def noise_report(df, name, batch_df):
    scored = df[df["manual_label"].notna()].copy()
    n = len(scored)
    correct = (scored["pred"] == scored["manual_label"]).sum()
    acc = correct / n
    lo, hi = wilson_ci(correct, n)
    print(f"\n{'='*64}")
    print(f"  {name}")
    print(f"{'='*64}")
    print(f"Reviewed: {n} | Agreement: {correct} ({acc*100:.1f}%) | Noise: {(1-acc)*100:.1f}%")
    print(f"95% CI (Wilson): [{lo*100:.1f}%, {hi*100:.1f}%]")
    print(f"\nPer-class noise (true class = manual_label):")
    per_class_rows = []
    for cls in ["skincare", "haircare", "makeup"]:
        mask = scored["manual_label"] == cls
        cmask = scored["manual_label"][mask] == scored["pred"][mask]
        if mask.sum() > 0:
            a = cmask.mean()
            per_class_rows.append({"class": cls, "n": int(mask.sum()), "acc": a})
            print(f"  {cls}: n={mask.sum()} acc={a*100:.1f}% noise={(1-a)*100:.1f}%")
    print(f"\nBatch size (all): {len(batch_df):,}")
    vc = batch_df["pred"].value_counts()
    print("Batch per-class:")
    for cls in ["skincare", "haircare", "makeup"]:
        print(f"  {cls}: {int(vc.get(cls, 0)):,}")
    print("\nProjected clean-example add per class (acc * volume):")
    for r in per_class_rows:
        if r["class"] in vc:
            clean = r["acc"] * vc[r["class"]]
            print(f"  {r['class']}: ~{clean:,.0f} clean of {int(vc[r['class']]):,}")
    return acc, lo, hi


print("### LOOSE BATCH")
if review_loose is not None:
    a_loose, *_ = noise_report(review_loose, "LOOSE", loose)
else:
    print("  no annotations yet — skipped")
print()
print("### CONSERVATIVE BATCH")
if review_cons is not None:
    a_cons, *_ = noise_report(review_cons, "CONSERVATIVE", conservative)
else:
    print("  no annotations yet — skipped")

### LOOSE BATCH

  LOOSE
Reviewed: 93 | Agreement: 88 (94.6%) | Noise: 5.4%
95% CI (Wilson): [88.0%, 97.7%]

Per-class noise (true class = manual_label):
  skincare: n=25 acc=96.0% noise=4.0%
  haircare: n=63 acc=95.2% noise=4.8%
  makeup: n=5 acc=80.0% noise=20.0%

Batch size (all): 10,592
Batch per-class:
  skincare: 7,061
  haircare: 2,317
  makeup: 1,214

Projected clean-example add per class (acc * volume):
  skincare: ~6,779 clean of 7,061
  haircare: ~2,207 clean of 2,317
  makeup: ~971 clean of 1,214

### CONSERVATIVE BATCH

  CONSERVATIVE
Reviewed: 168 | Agreement: 165 (98.2%) | Noise: 1.8%
95% CI (Wilson): [94.9%, 99.4%]

Per-class noise (true class = manual_label):
  skincare: n=74 acc=100.0% noise=0.0%
  haircare: n=85 acc=96.5% noise=3.5%
  makeup: n=9 acc=100.0% noise=0.0%

Batch size (all): 6,179
Batch per-class:
  skincare: 3,319
  haircare: 1,873
  makeup: 987

Projected clean-example add per class (acc * volume):
  skincare: ~3,319 clean of 3,319
  haircare: ~1,807 cl

## Threshold decision

Choose which batch to commit based on the full evidence: noise rate (aggregate AND per-class), volume,
and per-class spread. The conservative (strong-keyword) batch is usually the right call when label noise
feeds training, but pick based on the numbers above. Set `CHOSEN_BATCH` to `"conservative"`, `"loose"`,
or a manually-built blend below.

In [19]:
# Set which candidate batch becomes the auto-labeled set for v2.
CHOSEN_BATCH = "conservative"  # <-- change after reviewing step 3 numbers

if CHOSEN_BATCH == "conservative":
    auto = conservative.copy()
elif CHOSEN_BATCH == "loose":
    auto = loose.copy()
else:
    raise ValueError(f"Unknown CHOSEN_BATCH: {CHOSEN_BATCH}")

print(f"Chosen batch: {CHOSEN_BATCH} -> {len(auto):,} auto-labeled rows")
print(auto["pred"].value_counts().to_string())

Chosen batch: conservative -> 6,179 auto-labeled rows
pred
skincare    3319
haircare    1873
makeup       987


## 4. Commit the v1/v2 datasets

Build the two manifest CSVs (schemas with `label_source`) and populate the class image directories:

- `raw_v1_clean/`: the 10,215 manifest hand-labeled images only (v1 baseline, no strays, no auto).
- `raw_v2/`: the SAME 10,215 hand images **plus** the chosen auto-labeled batch's images.

This guarantees `raw_v1_clean` and `raw_v2` differ only by the auto-labeled batch.

In [20]:
manifest = pd.read_csv(MANIFEST_PATH, dtype={"code": str})
print(f"Hand-labeled manifest: {len(manifest):,} rows")
print(manifest["label"].value_counts().to_string())

hand = manifest.rename(columns={"label": "pred"}).copy()
hand["keyword"] = None
hand["strong"] = None
hand["label_source"] = "hand"

auto_rows = auto[["code", "imgid", "pred", "keyword", "strong"]].copy()
auto_rows["label_source"] = "auto_labeled"

cols = ["code", "imgid", "pred", "keyword", "strong", "label_source"]
v1_rows = hand[cols].copy()
v2_rows = pd.concat([hand[cols], auto_rows[cols]], ignore_index=True)

print(f"\nv1 (hand only): {len(v1_rows):,}")
print(f"v2 (hand + auto): {len(v2_rows):,}  (auto: {len(auto_rows):,})")

Hand-labeled manifest: 10,237 rows
label
skincare    6914
haircare    2504
makeup       819

v1 (hand only): 10,237
v2 (hand + auto): 16,416  (auto: 6,179)


In [21]:
auto_rows_loose = loose[["code", "imgid", "pred", "keyword", "strong"]].copy()
auto_rows_loose["label_source"] = "auto_labeled"

v3_rows = pd.concat([hand[cols], auto_rows_loose[cols]], ignore_index=True)

print(f"\nv3 (hand + loose auto): {len(v3_rows):,}  (auto: {len(auto_rows_loose):,})")


v3 (hand + loose auto): 20,829  (auto: 10,592)


In [22]:
def populate_dir(rows, dest_root, img_source_fn):
    """Copy images for each (code -> pred) into dest_root/<pred>/<code>.jpg."""
    dest_root.mkdir(parents=True, exist_ok=True)
    copied = 0
    missing = 0
    for pred in ["skincare", "haircare", "makeup"]:
        (dest_root / pred).mkdir(exist_ok=True)
    for _, r in rows.iterrows():
        src = img_source_fn(r["code"])
        dest = dest_root / r["pred"] / f"{r['code']}.jpg"
        if dest.exists():
            copied += 1
            continue
        if src is not None and src.exists():
            import shutil
            shutil.copy2(src, dest)
            copied += 1
        else:
            missing += 1
    return copied, missing


# Hand images live in data/raw/<class>/<code>.jpg
def hand_src(code):
    for cls in ["skincare", "haircare", "makeup"]:
        p = RAW_DIR / cls / f"{code}.jpg"
        if p.exists():
            return p
    return None

# Auto images live in data/unlabeled/<code>.jpg
def auto_src(code):
    p = UNLABELED_DIR / f"{code}.jpg"
    return p if p.exists() else None

import time

print("Populating raw_v1_clean (hand only)...")
t0 = time.time()
v1_copied, v1_missing = populate_dir(v1_rows, V1_CLEAN_DIR, hand_src)
print(f"  copied {v1_copied:,} / {len(v1_rows):,}, missing {v1_missing:,} ({time.time()-t0:.0f}s)")

print("Populating raw_v2 (hand + auto)...")
t0 = time.time()
v2_copied, v2_missing = populate_dir(v2_rows, V2_DIR, lambda code: hand_src(code) or auto_src(code))
print(f"  copied {v2_copied:,} / {len(v2_rows):,}, missing {v2_missing:,} ({time.time()-t0:.0f}s)")

Populating raw_v1_clean (hand only)...
  copied 10,233 / 10,237, missing 4 (2s)
Populating raw_v2 (hand + auto)...
  copied 16,412 / 16,416, missing 4 (4s)


In [23]:
print("Populating raw_v3_loose (hand only + loose auto)...")
t0 = time.time()
v3_copied, v3_missing = populate_dir(v3_rows, V3_LOOSE_DIR, lambda code: hand_src(code) or auto_src(code))
print(f"  copied {v3_copied:,} / {len(v3_rows):,}, missing {v3_missing:,} ({time.time()-t0:.0f}s)")

Populating raw_v3_loose (hand only + loose auto)...
  copied 20,825 / 20,829, missing 4 (22s)


In [24]:
# Write final manifest CSVs (label_source-tagged) to backend/data/manifests/.
out_cols = ["code", "imgid", "pred", "keyword", "strong", "label_source"]
v1_rows[out_cols].to_csv(MANIFESTS_DIR / "train_manifest_v1_clean.csv", index=False)
v2_rows[out_cols].to_csv(MANIFESTS_DIR / "train_manifest_v2_autolabeled.csv", index=False)
v3_rows[out_cols].to_csv(MANIFESTS_DIR / "train_manifest_v3_loose.csv", index=False)
print(f"  {MANIFESTS_DIR / 'train_manifest_v3_loose.csv'}")

print("\nFinal v3 manifest label_source counts:")
print(v3_rows["label_source"].value_counts().to_string())
print("\nFinal v3 manifest per-class (auto only):")
print(auto_rows_loose["pred"].value_counts().to_string())

print(f"Wrote:\n  {MANIFESTS_DIR / 'train_manifest_v1_clean.csv'}\n  {MANIFESTS_DIR / 'train_manifest_v2_autolabeled.csv'}")

print("\nFinal v2 manifest label_source counts:")
print(v2_rows["label_source"].value_counts().to_string())
print("\nFinal v2 manifest per-class (auto only):")
print(auto_rows["pred"].value_counts().to_string())

  c:\Projects\cosmetic-expiry-scanner\backend\data\manifests\train_manifest_v3_loose.csv

Final v3 manifest label_source counts:
label_source
auto_labeled    10592
hand            10237

Final v3 manifest per-class (auto only):
pred
skincare    7061
haircare    2317
makeup      1214
Wrote:
  c:\Projects\cosmetic-expiry-scanner\backend\data\manifests\train_manifest_v1_clean.csv
  c:\Projects\cosmetic-expiry-scanner\backend\data\manifests\train_manifest_v2_autolabeled.csv

Final v2 manifest label_source counts:
label_source
hand            10237
auto_labeled     6179

Final v2 manifest per-class (auto only):
pred
skincare    3319
haircare    1873
makeup       987


In [25]:
# Final verification: image counts per class dir in both datasets.
def dir_counts(root):
    out = {}
    for cls in ["skincare", "haircare", "makeup"]:
        d = root / cls
        out[cls] = sum(1 for p in d.glob("*.jpg")) if d.exists() else 0
    return out

v1c = dir_counts(V1_CLEAN_DIR)
v2c = dir_counts(V2_DIR)
v3c = dir_counts(V3_LOOSE_DIR)
print("\nraw_v3_loose:")
for c, n in v3c.items():
    print(f"  {c}: {n:,}")
print(f"  TOTAL: {sum(v3c.values()):,}")

print("\nDelta (v3 - v1) per class:")
for c in v1c:
    print(f"  {c}: +{v3c[c] - v1c[c]:,}")
print("raw_v1_clean:")
for c, n in v1c.items():
    print(f"  {c}: {n:,}")
print(f"  TOTAL: {sum(v1c.values()):,}")
print("\nraw_v2:")
for c, n in v2c.items():
    print(f"  {c}: {n:,}")
print(f"  TOTAL: {sum(v2c.values()):,}")

# Sanity: how much did v2 add per class vs v1?
print("\nDelta (v2 - v1) per class:")
for c in v1c:
    print(f"  {c}: +{v2c[c] - v1c[c]:,}")


raw_v3_loose:
  skincare: 13,974
  haircare: 4,818
  makeup: 2,033
  TOTAL: 20,825

Delta (v3 - v1) per class:
  skincare: +7,061
  haircare: +2,317
  makeup: +1,214
raw_v1_clean:
  skincare: 6,913
  haircare: 2,501
  makeup: 819
  TOTAL: 10,233

raw_v2:
  skincare: 13,974
  haircare: 4,818
  makeup: 2,033
  TOTAL: 20,825

Delta (v2 - v1) per class:
  skincare: +7,061
  haircare: +2,317
  makeup: +1,214


## Usage

Flip `DATASET_VERSION` in `train_classifier.ipynb`'s ablation toggle cell to `"v1"` (clean baseline,
`raw_v1_clean`) or `"v2"` (hand + auto, `raw_v2`) and re-run training. Because the two datasets differ
only by the auto-labeled batch, the train-dev error delta isolates the volume auto-labeling added,
separate from the contamination fix.